# Day 2 — Sequence Modeling, POS Tagging and Language Models
## 30 Days of AI: From NLP to LLMs

---

This is the conceptual bridge from classical NLP to RNNs and
Transformers. Everything you learn today — sequence labeling, n-gram
models, the Markov assumption, perplexity — forms the foundation that
motivated the design of modern neural architectures.

---

### What You Will Learn Today

- Why sequence modeling matters and where it differs from bag-of-words
- Part-of-Speech tagging as a sequence labeling problem
- N-gram language models: unigram, bigram, trigram
- The Markov assumption and why it is both useful and limited
- Data sparsity and smoothing techniques
- Perplexity as a language model evaluation metric
- The transition from statistical to neural language models

### Goal by End of Day

Build a POS tagger using NLTK and a bigram language model from scratch
that can score sentence probabilities and generate simple text.

---

## Part 1 — Why Sequence Modeling Matters

Word order carries meaning. Bag-of-words models completely ignore this.
```
"dog bites man"   ← very different meaning
"man bites dog"   ← same words, completely different event
```

Sequence modeling captures:

- Word order and position
- Context dependencies between words
- Grammar patterns and syntax structure
- Long-range relationships (a word at position 1 affecting position 20)

Every powerful NLP model — RNNs, LSTMs, Transformers — is fundamentally
a sequence model. Today we start with the statistical foundations before
moving to neural approaches in the coming days.

---

## Part 2 — Part-of-Speech Tagging

POS tagging assigns a grammatical role to every token in a sentence.
```
Sentence  :  "I    love    NLP    every    day"
POS Tags  :  PRON  VERB    NOUN   DET      NOUN
```

Common POS tags in the Penn Treebank tagset:

| Tag | Meaning | Example |
|-----|---------|---------|
| NN | Noun, singular | dog, city |
| NNS | Noun, plural | dogs, cities |
| VB | Verb, base form | run, eat |
| VBD | Verb, past tense | ran, ate |
| JJ | Adjective | quick, blue |
| RB | Adverb | quickly, very |
| PRP | Personal pronoun | I, he, she |
| DT | Determiner | the, a, an |
| IN | Preposition | in, on, at |
| CC | Coordinating conjunction | and, but, or |

### Why POS Tagging Matters Downstream

POS tagging is not just an academic exercise. It improves nearly every
downstream NLP task:

- **Named Entity Recognition** — knowing a word is a proper noun (NNP)
  helps identify it as an entity
- **Information Extraction** — verb phrases signal actions and events
- **Grammar Correction** — subject-verb agreement requires knowing POS
- **Machine Translation** — word order rules differ by language and
  depend on grammatical roles
- **Question Answering** — identifying the verb in a question reveals
  what type of answer is expected

> Interview insight: POS tagging is a sequence labeling problem, not a
> classification problem. The label of each token depends not just on
> the token itself but on the labels of its neighbors.

---

## Part 3 — Sequence Labeling Intuition

POS tagging and Named Entity Recognition are both **sequence labeling**
problems. The key distinction from simple classification:

In classification, each input is labeled independently.
In sequence labeling, labels depend on each other.
```
"New    York    is    a    city"
 NNP    NNP     VBZ   DT   NN

"New" alone could be adjective (JJ)
"New" followed by "York" must be NNP (proper noun)
```

The model must consider:

- The current token and its features
- The surrounding context tokens
- The dependency between adjacent labels (NNP often follows NNP)
- Grammar constraints (an adjective is unlikely to follow a verb directly)

This label dependency is why sequence models — HMMs, CRFs, LSTMs,
Transformers — outperform simple per-token classifiers on tagging tasks.

---

## Part 4 — N-gram Language Models

A language model assigns a probability to any sequence of words.
This is the foundational idea behind everything from autocomplete
to ChatGPT.
```
P("I love NLP")  =  probability this exact sequence appears in language
```

### Unigram Model

Assumes every word is independent of all others.
```
P("I love NLP")  =  P("I") × P("love") × P("NLP")
```

Simple and fast. Completely ignores word order and context.
"NLP love I" would get the exact same probability.

### Bigram Model

Conditions each word on the one word before it.
```
P("I love NLP")  =  P("I") × P("love" | "I") × P("NLP" | "love")
```

Captures local context. "love" is more likely after "I" than after "the".

### Trigram Model

Conditions each word on the two words before it.
```
P("I love NLP")  =  P("I") × P("love" | "I") × P("NLP" | "I", "love")
```

Richer context but requires much more data to estimate reliably.

> Interview insight: Higher n captures more context but creates a data
> sparsity problem. A trigram like ("machine", "learning", "is") may
> never appear in your training corpus even if each word is common.
> This is why n-gram models top out around trigrams or 4-grams in practice.

---

## Part 5 — The Markov Assumption

N-gram models are built on the **Markov assumption**: the probability
of the next word depends only on the last n-1 words, not the entire
history.
```
Full dependency  :  P(wt | w1, w2, ..., wt-1)   ← intractable
Markov (bigram)  :  P(wt | wt-1)                 ← practical
Markov (trigram) :  P(wt | wt-2, wt-1)           ← more context
```

This simplification makes probability estimation tractable from finite
training data. But it is also the fundamental limitation of n-gram
models — real language has long-range dependencies that far exceed
any practical window size.
```
"The trophy didn't fit in the suitcase because it was too big."

What does "it" refer to? The trophy.
A bigram model sees only the previous word ("big") — no help.
A trigram model sees ("was", "too") — still no help.
You need to go back 8 words. This is why RNNs and Transformers exist.
```

---

## Part 6 — Data Sparsity and Smoothing

The data sparsity problem: most word combinations never appear in
training data, even in very large corpora. An n-gram that was never
seen gets probability zero — and multiplying any probability chain
by zero collapses the entire sentence probability to zero.

**Laplace Smoothing (Add-1)**
Add 1 to every count, including unseen combinations.
Simple but over-allocates probability to unseen events.
```
P(w2 | w1)  =  (count(w1, w2) + 1) / (count(w1) + V)
where V = vocabulary size
```

**Kneser-Ney Smoothing**
The gold standard for n-gram models. Redistributes probability mass
from seen to unseen n-grams based on how versatile a word is as a
context word, not just how frequent it is.

**Backoff Models**
If a trigram count is zero, fall back to the bigram.
If the bigram is also zero, fall back to the unigram.
Always have a non-zero estimate.

> Interview insight: Smoothing is not optional — it is essential.
> Without it, a single unseen word makes the probability of any
> sentence containing it exactly zero, making the model useless
> for any real text.

---

## Part 7 — Perplexity

Perplexity is the standard evaluation metric for language models.
It measures how surprised the model is by the test text.
```
Perplexity  =  2 ^ (- (1/N) × Σ log2 P(wi | context))

Lower perplexity  →  model is less surprised  →  better model
```

Intuitive interpretation: a perplexity of 100 means the model is as
confused as if it had to choose uniformly among 100 words at every step.
A perplexity of 10 means it effectively considers only 10 words at each
position — much more confident and accurate.
```
Random model on English     :  perplexity ≈ 50,000  (full vocab)
Trigram model on news text  :  perplexity ≈ 60-100
Good neural LM              :  perplexity ≈ 10-30
GPT-4 class models          :  perplexity ≈ 3-8
```

> Interview insight: Perplexity is to language models what accuracy
> is to classifiers — but lower is better. It measures the geometric
> mean inverse probability the model assigns to held-out text.

---

## Part 8 — From Statistical to Neural Language Models

N-gram models have served NLP well for decades but have hard limits:

| Limitation | Explanation |
|---|---|
| Fixed context window | Cannot capture dependencies beyond n words |
| Data sparsity | Rare combinations always problematic |
| No generalization | "cat" and "feline" are completely unrelated |
| Storage cost | Storing all n-gram counts requires huge memory |

Neural language models solve all of these:

- **RNNs** — process sequences step by step, theoretically unlimited memory
- **LSTMs** — gated RNNs that selectively remember long-range context
- **Transformers** — attend to all positions simultaneously, no sequential bottleneck

The fundamental goal is the same: predict the next word given context.
Only the mechanism changes. Tomorrow we build the RNN version.

In [2]:
import re
import math
import string
import random
from collections import defaultdict, Counter

import nltk
nltk.download('punkt',quiet=True)
nltk.download('punkt_tab',quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords',quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag


# ── Part 1: POS Tagger ────────────────────────────────────────────────

sentences = [
    "The quick brown fox jumps over the lazy dog",
    "Natural language processing enables machines to understand text",
    "I love learning about artificial intelligence every single day",
    "The president signed the new climate bill into law yesterday",
    "She carefully analyzed the experimental results from the study",
]

print("POS Tagging Results")
print("=" * 65)

for sentence in sentences:
    tokens = word_tokenize(sentence)
    tags   = pos_tag(tokens)
    print(f"\nSentence : {sentence}")
    print(f"{'Token':<18} {'POS Tag':<10} {'Meaning'}")
    print("-" * 50)

    tag_meanings = {
        'NN': 'Noun singular',    'NNS': 'Noun plural',
        'NNP': 'Proper noun',     'NNPS': 'Proper noun plural',
        'VB': 'Verb base',        'VBD': 'Verb past tense',
        'VBG': 'Verb gerund',     'VBN': 'Verb past participle',
        'VBP': 'Verb present',    'VBZ': 'Verb 3rd person',
        'JJ': 'Adjective',        'JJR': 'Adjective comparative',
        'JJS': 'Adjective superlative',
        'RB': 'Adverb',           'RBR': 'Adverb comparative',
        'PRP': 'Pronoun personal', 'PRP$': 'Pronoun possessive',
        'DT': 'Determiner',       'IN': 'Preposition',
        'CC': 'Conjunction',      'CD': 'Cardinal number',
        'TO': 'to',               'MD': 'Modal verb',
    }

    for token, tag in tags:
        meaning = tag_meanings.get(tag, tag)
        print(f"  {token:<16} {tag:<10} {meaning}")

# ── Part 2: POS Tag Distribution Analysis ────────────────────────────

corpus_text = " ".join(sentences)
all_tokens  = word_tokenize(corpus_text)
all_tags    = pos_tag(all_tokens)

tag_counts = Counter(tag for _, tag in all_tags)

print("POS Tag Distribution in Corpus")
print("=" * 45)
for tag, count in tag_counts.most_common(12):
    bar = "#" * (count * 4)
    print(f"  {tag:<8} {count:>3}  {bar}")

# ── Part 3: N-gram Language Model from Scratch ───────────────────────

training_corpus = [
    "i love natural language processing",
    "natural language processing is fascinating",
    "machine learning powers modern nlp",
    "deep learning models understand language well",
    "transformers changed natural language processing forever",
    "language models predict the next word",
    "word embeddings capture semantic meaning",
    "nlp enables machines to understand human language",
    "i enjoy learning about machine learning every day",
    "sequence models process text word by word",
    "the model learns patterns from training data",
    "attention mechanisms help models focus on context",
    "language is inherently sequential and context dependent",
    "grammar and syntax define the structure of language",
    "modern nlp models are trained on large text corpora",
]

def tokenize_corpus(corpus):
    tokenized = []
    for sentence in corpus:
        tokens = ["<START>"] + sentence.lower().split() + ["<END>"]
        tokenized.append(tokens)
    return tokenized

tokenized = tokenize_corpus(training_corpus)

print("Tokenized Corpus Sample")
print("=" * 55)
for sent in tokenized[:3]:
    print(" ", sent)

# ── Build Unigram Model ───────────────────────────────────────────────

def build_unigram(tokenized_corpus):
    counts = Counter()
    for sentence in tokenized_corpus:
        counts.update(sentence)
    total = sum(counts.values())
    probs = {word: count / total for word, count in counts.items()}
    return counts, probs

unigram_counts, unigram_probs = build_unigram(tokenized)

print("Unigram Probabilities — Top 15 Words")
print("=" * 45)
for word, prob in sorted(
    unigram_probs.items(), key=lambda x: -x[1]
)[:15]:
    bar = "#" * int(prob * 200)
    print(f"  {word:<18} {prob:.4f}  {bar}")

# ── Build Bigram Model ────────────────────────────────────────────────

def build_bigram(tokenized_corpus, vocab_size, alpha=1.0):
    """
    Builds a bigram model with Laplace smoothing.
    alpha = smoothing parameter (1.0 = add-1 / Laplace)
    """
    unigram_counts = Counter()
    bigram_counts  = defaultdict(Counter)

    for sentence in tokenized_corpus:
        unigram_counts.update(sentence)
        for i in range(len(sentence) - 1):
            bigram_counts[sentence[i]][sentence[i+1]] += 1

    def bigram_prob(w1, w2):
        numerator   = bigram_counts[w1][w2] + alpha
        denominator = unigram_counts[w1] + alpha * vocab_size
        return numerator / denominator

    return bigram_counts, unigram_counts, bigram_prob

vocabulary = set(
    word for sentence in tokenized for word in sentence
)
vocab_size = len(vocabulary)

bigram_counts, uni_counts, bigram_prob = build_bigram(
    tokenized, vocab_size
)

print(f"Vocabulary size : {vocab_size} unique tokens")
print()

# Show sample bigram probabilities
print("Sample Bigram Probabilities  P(w2 | w1)")
print("=" * 50)
sample_pairs = [
    ("natural",  "language"),
    ("language", "processing"),
    ("machine",  "learning"),
    ("i",        "love"),
    ("i",        "hate"),       # never seen — smoothed
    ("deep",     "learning"),
    ("<START>",  "i"),
]

for w1, w2 in sample_pairs:
    prob = bigram_prob(w1, w2)
    print(f"  P({w2:<14} | {w1:<12})  =  {prob:.6f}")

# ── Sentence Probability Scoring ──────────────────────────────────────

def sentence_probability(sentence, bigram_prob_fn, log=False):
    """
    Computes the probability of a sentence under the bigram model.
    Uses log probabilities to avoid underflow on long sentences.
    """
    tokens    = ["<START>"] + sentence.lower().split() + ["<END>"]
    log_prob  = 0.0

    for i in range(len(tokens) - 1):
        p = bigram_prob_fn(tokens[i], tokens[i+1])
        log_prob += math.log2(p) if p > 0 else float('-inf')

    if log:
        return log_prob
    return 2 ** log_prob

test_sentences = [
    "i love natural language processing",    # in-distribution
    "machine learning is fascinating",       # in-distribution
    "language models predict the next word", # in-distribution
    "cat sat potato jumping purple",         # nonsense
    "the quick brown fox",                   # reasonable but unseen
]

print("Sentence Probability Scoring — Bigram Model")
print("=" * 60)
for sent in test_sentences:
    prob     = sentence_probability(sent, bigram_prob)
    log_prob = sentence_probability(sent, bigram_prob, log=True)
    print(f"\nSentence  : {sent}")
    print(f"Prob      : {prob:.2e}")
    print(f"Log Prob  : {log_prob:.4f}")

# ── Perplexity ────────────────────────────────────────────────────────

def compute_perplexity(test_corpus, bigram_prob_fn):
    """
    Perplexity = 2 ^ (- (1/N) * sum of log2 probabilities)
    Lower is better.
    """
    log_prob_sum = 0.0
    token_count  = 0

    for sentence in test_corpus:
        tokens = ["<START>"] + sentence.lower().split() + ["<END>"]
        for i in range(len(tokens) - 1):
            p = bigram_prob_fn(tokens[i], tokens[i+1])
            if p > 0:
                log_prob_sum += math.log2(p)
            else:
                log_prob_sum += float('-inf')
            token_count += 1

    avg_log_prob = log_prob_sum / token_count
    perplexity   = 2 ** (-avg_log_prob)
    return perplexity

# Evaluate on in-distribution and out-of-distribution text
in_distribution = [
    "i love natural language processing",
    "machine learning models understand language",
    "language is inherently sequential",
]

out_of_distribution = [
    "the cat sat on the mat",
    "stock markets fell sharply today",
    "she baked a chocolate cake yesterday",
]

ppl_in  = compute_perplexity(in_distribution,  bigram_prob)
ppl_out = compute_perplexity(out_of_distribution, bigram_prob)

print("Perplexity Evaluation")
print("=" * 45)
print(f"In-distribution text     :  perplexity = {ppl_in:.2f}")
print(f"Out-of-distribution text :  perplexity = {ppl_out:.2f}")
print()
print("Lower perplexity = model is less surprised = text is more")
print("consistent with what it learned from the training corpus.")

# ── Trigram Model ─────────────────────────────────────────────────────

def build_trigram(tokenized_corpus, vocab_size, alpha=1.0):
    bigram_counts  = defaultdict(Counter)
    trigram_counts = defaultdict(Counter)

    for sentence in tokenized_corpus:
        for i in range(len(sentence) - 1):
            bigram_counts[sentence[i]][sentence[i+1]] += 1
        for i in range(len(sentence) - 2):
            context = (sentence[i], sentence[i+1])
            trigram_counts[context][sentence[i+2]] += 1

    def trigram_prob(w1, w2, w3):
        context     = (w1, w2)
        numerator   = trigram_counts[context][w3] + alpha
        denominator = sum(bigram_counts[w1].values()) + alpha * vocab_size
        return numerator / denominator

    return trigram_counts, trigram_prob

trigram_counts, trigram_prob = build_trigram(tokenized, vocab_size)

print("Sample Trigram Probabilities  P(w3 | w1, w2)")
print("=" * 60)
trigram_samples = [
    ("natural",  "language", "processing"),
    ("machine",  "learning", "models"),
    ("i",        "love",     "natural"),
    ("deep",     "learning", "models"),
    ("i",        "love",     "pizza"),     # never seen
]

for w1, w2, w3 in trigram_samples:
    prob = trigram_prob(w1, w2, w3)
    print(f"  P({w3:<14} | {w1:<10} {w2:<12})  =  {prob:.6f}")

# ── Text Generation with Bigram Model ────────────────────────────────

def generate_text(bigram_counts, unigram_counts, seed_word,
                  max_words=12, temperature=1.0):
    """
    Generates text by sampling from the bigram distribution.
    Temperature > 1 = more random. Temperature < 1 = more deterministic.
    """
    result  = [seed_word]
    current = seed_word

    for _ in range(max_words):
        if current not in bigram_counts:
            break

        candidates = bigram_counts[current]
        if not candidates:
            break

        words  = list(candidates.keys())
        counts = np.array(list(candidates.values()), dtype=float)

        # Apply temperature
        counts = counts ** (1.0 / temperature)
        probs  = counts / counts.sum()

        next_word = np.random.choice(words, p=probs)

        if next_word == "<END>":
            break

        result.append(next_word)
        current = next_word

    return " ".join(result)

import numpy as np
np.random.seed(42)

print("Text Generation — Bigram Model")
print("=" * 55)

seed_words   = ["i", "natural", "machine", "language", "deep"]
temperatures = [0.5, 1.0, 1.5]

for seed in seed_words:
    print(f"\nSeed word : '{seed}'")
    for temp in temperatures:
        generated = generate_text(
            bigram_counts, uni_counts,
            seed, max_words=10, temperature=temp
        )
        print(f"  temp={temp}  :  {generated}")

# ── Backoff Language Model ────────────────────────────────────────────

def backoff_probability(w1, w2, w3,
                        trigram_prob_fn, bigram_prob_fn,
                        alpha=0.4):
    """
    Stupid backoff: try trigram first, fall back to bigram if zero.
    alpha penalizes the lower-order model.
    """
    tri_p = trigram_prob_fn(w1, w2, w3)
    if tri_p > 0:
        return "trigram", tri_p

    bi_p = bigram_prob_fn(w2, w3)
    if bi_p > 0:
        return "bigram", alpha * bi_p

    return "uniform", alpha ** 2 * (1.0 / vocab_size)

print("Backoff Model — Demonstrating Graceful Degradation")
print("=" * 60)
backoff_tests = [
    ("natural",  "language", "processing"),  # seen trigram
    ("machine",  "learning", "is"),          # may need backoff
    ("random",   "unseen",   "trigram"),     # definitely needs backoff
]

for w1, w2, w3 in backoff_tests:
    source, prob = backoff_probability(
        w1, w2, w3, trigram_prob, bigram_prob
    )
    print(f"\n  P({w3} | {w1}, {w2})")
    print(f"  Source : {source}")
    print(f"  Prob   : {prob:.6f}")

# ── Comparing Unigram vs Bigram vs Trigram ────────────────────────────

comparison_sentences = [
    "i love natural language processing",
    "language models predict the next word",
    "cat potato purple jumping sky",          # nonsense
]

print("Model Comparison — Sentence Log Probabilities")
print("=" * 65)
print(f"{'Sentence':<42} {'Unigram':>10}  {'Bigram':>10}")
print("-" * 65)

for sent in comparison_sentences:
    tokens = sent.lower().split()

    # Unigram log prob
    uni_lp = sum(
        math.log2(unigram_probs.get(t, 1e-10))
        for t in tokens
    )

    # Bigram log prob
    tok_b  = ["<START>"] + tokens + ["<END>"]
    bi_lp  = sum(
        math.log2(bigram_prob(tok_b[i], tok_b[i+1]))
        for i in range(len(tok_b) - 1)
    )

    label = sent[:40] + ".." if len(sent) > 40 else sent
    print(f"{label:<42} {uni_lp:>10.3f}  {bi_lp:>10.3f}")

print()
print("More negative = lower probability = model thinks this")
print("sentence is less likely. Nonsense gets the lowest score.")

POS Tagging Results

Sentence : The quick brown fox jumps over the lazy dog
Token              POS Tag    Meaning
--------------------------------------------------
  The              DT         Determiner
  quick            JJ         Adjective
  brown            NN         Noun singular
  fox              NN         Noun singular
  jumps            VBZ        Verb 3rd person
  over             IN         Preposition
  the              DT         Determiner
  lazy             JJ         Adjective
  dog              NN         Noun singular

Sentence : Natural language processing enables machines to understand text
Token              POS Tag    Meaning
--------------------------------------------------
  Natural          JJ         Adjective
  language         NN         Noun singular
  processing       NN         Noun singular
  enables          VBZ        Verb 3rd person
  machines         NNS        Noun plural
  to               TO         to
  understand       VB         Verb base

## Day 5 Summary

| Concept | Core Idea | Limitation |
|---|---|---|
| **Sequence Modeling** | Word order and context matter | Bag-of-words ignores both |
| **POS Tagging** | Assign grammatical roles to tokens | Labels depend on neighbors |
| **Unigram Model** | Words are independent | Ignores all context |
| **Bigram Model** | Each word depends on previous one | Short memory |
| **Trigram Model** | Each word depends on previous two | Data sparsity grows |
| **Markov Assumption** | Only last n words matter | Real language has longer dependencies |
| **Laplace Smoothing** | Add 1 to all counts | Over-smooths rare events |
| **Backoff** | Fall back to lower-order model | Loses some context |
| **Perplexity** | Measures model surprise on test data | Lower is better |

*30 Days of AI — Day 5 of 30 | Sequence Modeling, POS Tagging and Language Models*